# 77 · The human vs mouse calls of notebook 71, downstream

Decision D5 makes notebook 71's positional F-test calls (83 human vs mouse sets) the paper's species list; they
replace the retired 22-pathway list. This notebook reruns, on those calls, the analyses that described the 22:

| Part | Question | Old source |
|---|---|---|
| 3.2 | What do the calls summarise, by symmetric zonation class? | 43 §7, 50 |
| 3.3 | In which segment does each call differ most, and which species is higher there? | 47 (e), 50 |
| 3.4 | How far do a coordinate-free screen and the no-refit robustness checks support the calls? | 50, 47 (e), R5 |
| 3.5 | Do the calls replicate in external atlases beyond matched random sets and beyond noise calls? | 48 A, 50, 37 E3 |
| 3.6 | Do the conventional-only sets replicate their average difference once overlap is respected? | 50 part 3 |
| 3.7 | The lead genes: ranks, segment values, verdicts, curves | 37, 38, 41 |

The protocol is `docs/results/pt-package-species-calls.md`, frozen before this notebook ran, with the coordinator's
decisions O1–O6. Notebook 72's verdicts bound the wording: species registration is unstable (C8 not met) and the
coordinate is noise-dominated within segments (C14 not met), so every location is reported at segment resolution
only. Nothing here refits a gene model. Cohort: two male control mice and two cortex sections of one male human donor;
everything is descriptive and specimens are the replicates. Human vs mouse only.

## 1 · Setup

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, os.environ.get('PSEUDOSPACE_N_JOBS', '24'))   # thread cap; never in a cache key
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
external = data_root / 'external'
package_root = results_root / 'package_pseudospace'
EXPORT = package_root / 'hvm' / 'pt_units.h5ad'
NATIVE_COR1 = package_root / 'runs' / 'hvm' / '01_units' / 'HUK1_COR1_units.h5ad'
upstream13 = results_root / 'minimal_pt_scfates'                  # the ortholog map
T71 = results_root / 'pt_pathways_package_pseudospace'            # read only: tables and stage cache
T72 = results_root / 'pt_package_coordinate_validation'
T73 = results_root / 'pt_package_label_comparison'
T74 = results_root / 'pt_package_continuous_vs_discrete'
T75 = results_root / 'pt_package_pathway_methods'
nb38 = results_root / 'pt_literature_deep'
CENSUS = external / 'census_pt_segments'
OLD = {'37': results_root / 'pt_pathway_final' / 'scfates', '38': nb38, '43': results_root / 'pt_revision_classes' / 'reviewed',
       '47': results_root / 'pt_revision_addenda', '50': results_root / 'pt_primary_pathways',
       '12': results_root / 'pt_pathway_remodeling_scfates', '69': results_root / 'pt_pathways_without_mito'}
output = results_root / 'pt_package_species_calls'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
DS = 'human_vs_mouse'                                             # notebook 71's dataset name
PARTITION_FILES = {'condition': 'species', 'relabeling 1': 'swap_HUK1_COR1_Ctrl1A2', 'relabeling 2': 'swap_HUK1_COR1_Ctrl1A4'}
REQUIRED = [EXPORT, NATIVE_COR1, upstream13 / 'ortholog_map_used.csv',
            *[T71 / 'tables' / f for f in ('inputs.csv', f'universe_{DS}.csv', f'gene_sets_{DS}.json', f'gene_statistics_{DS}.csv',
                                           f'redundancy_clusters_{DS}.csv', 's9_winsor_lost_new.csv',
                                           *[f'pathway_tests_{DS}_cameraPR_{v}.csv' for v in PARTITION_FILES.values()])],
            T72 / 'tables' / 'transitions.csv', T72 / 'tables' / 'criteria_hvm.csv', T73 / 'run_record.json',
            T73 / 'tables' / '43' / 'gene_classes_symmetric.csv', T73 / 'tables' / 'mt_in_library' / '43' / 'gene_classes_symmetric.csv',
            *[T75 / 'tables' / f for f in (f'gene_statistics_segment_{DS}.csv', f'pathway_tests_segment_{DS}.csv',
                                           f'pathway_tests_segment_joint_{DS}.csv', f'pathway_tests_raw_tspatial_{DS}.csv',
                                           f'pathway_tests_deseq2_wholept_gsea_{DS}.csv', f'pathway_tests_deseq2_segments_gsea_{DS}.csv',
                                           f'deseq2_genes_{DS}.csv', f'programs_{DS}.csv')],
            *[CENSUS / f'{s}_pt_segment_pseudobulk_{k}' for s in ('human', 'mouse') for k in ('counts.csv.gz', 'meta.csv')],
            *[nb38 / f for f in ('probe_counts.csv', 'story_gene_evidence.csv', 'story_verdicts.csv', 'gene_level_lake_contrasts.csv',
                                 'lake_cortex_segment_means.csv')],
            *[external / 'xiong2023_sex_bias' / f'PT-{s}_fWTvsmWT.csv' for s in ('S1', 'S2', 'S3')],
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz',
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 38, 70–73 and 75 first)')

NOTEBOOK_LOGIC_VERSION = '77.package_species_calls.1'   # Bump when a cached calculation changes.
FINGERPRINT = 'efafc9ac2e99'                            # confirmed hvm pass-1 labels (notebook 70)
ALPHA = .05
DF1, DF2 = 6, 12                                        # notebook 71's positional F-test (guard)
TOP_SHARE, PROGRAM_CUT = .10, .5                        # contributing genes and programs (notebook 75's rule)
N_PERM, SEED, GENE_SEED = 9999, 50, 12                  # collection nulls (notebook 50), gene-level draws (notebook 37)
MIN_CELLS = 50                                          # external donors (notebooks 37, 50)
CHECK_Q, CHECK_RATIO, DETECTION_BOTH = .10, .7, .05     # notebook 37's gene-subset checks
FLAGGED_GENES = ['Alb', 'Ttr', 'Apoa1', 'Apoa2', 'Apoh', 'Fgb', 'Fga', 'Fgg', 'Egf', 'Muc1', 'Klk1', 'Slc26a7', 'Car12',
                 'Akr1c18', 'Ugt1a1', 'Ugt1a9']        # notebook 37
PROTEASOME = ['Psma3', 'Psma4', 'Psmb5', 'Psmb6', 'Psmc2', 'Psmc6', 'Psmd11', 'Psmd14']   # notebook 71
LEAD_GENES = ['Gatm', 'Gamt', 'Acadm', 'Acaa2', 'Dcxr', 'Ugt3a1', 'Gclc', 'Cyp2e1', 'Slc7a13', 'Rbp4']   # 41's nine + Rbp4 (R6)
LEAD_STORIES = ['creatine', 'fao', 'detox', 'reversal']
STORY_ORDER = ['creatine', 'fao', 'detox', 'reversal', 'sterol', 'lost_in_mouse', 'mouse_only_lipid', 'early_program_later',
               'oat_illustration', 'probe_example']
SEGMENTS = ('S1', 'S2', 'S3')
HEAT_PER_SEGMENT, MAX_ROWS, Z_CLIP, BINS_LEAD = 4, 40, 2.5, 20
NAMES = ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1']
MOUSE, HUMAN = NAMES[:2], NAMES[2:]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
MOUSE_COLOR, HUMAN_COLOR = '#0072B2', '#D55E00'
CLASS_COLORS = {'conserved': '#009E73', 'reversal': '#CC79A7', 'mouse-only': MOUSE_COLOR, 'human-only': HUMAN_COLOR,
                'indeterminate': '#E3E3E3', 'neither': '#B8B8B8'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 120)
READING, GUARDS = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)
READING.append(f"Logic version {NOTEBOOK_LOGIC_VERSION!r}, by the coordinator's phase-2 instruction (the protocol named "
               "'77.species_calls.1').")

## 2 · Inputs and guards

The inputs are built as notebook 71 builds them (`package_inputs.hvm_inputs`, the 13 ortholog-map `mt-` genes out
of the library and the universe). Guards (not cached): G1 export contract and the confirmed pass-1 fingerprint;
71's universe, sets and units. G2: 71's cached condition model reproduces its gene table (T_spatial to 10⁻¹², z
through `moderated_f`). G3: the 83 calls are 71's, and `overlap_programs` on them reproduces 75's programs. G4: the
logistic transitions reproduce 72's. G5: 75's segment tables cover 71's universe and sets; 74's tables are read when
present. G6: 73's classes come from the run with the confirmed fingerprint. G7 (section 7): the Lake pseudobulk
equals notebook 50's on the genes both share.

In [ ]:
from scipy import sparse

from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.package_inputs import hvm_inputs, read_pt_units
from pseudospace.package_labels import harmony_features
from pseudospace.pathway_decoys import overlap_programs
from pseudospace.pathway_pipelines import camera_pr, common_grid, moderated_f
from pseudospace.stage_cache import cached_payload, digest
from pseudospace import species_calls as sc
from pseudospace import set_collection_null

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
assert len(MT) == 13, MT
UNITS = read_pt_units(EXPORT, 'hvm')                                                       # G1: export contract
PROV = UNITS.uns['provenance']
assert str(PROV['notebook_logic_version']).startswith('70.'), PROV['notebook_logic_version']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, PROV['fingerprints']['pass1']
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES = list(INP['genes'])
SETS = INP['gene_sets']
universe71 = pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv')
if 'eligible' in universe71:
    universe71 = universe71[universe71.eligible.astype(bool)]
assert GENES == universe71.gene.astype(str).tolist(), 'G1: the universe differs from notebook 71'
assert SETS == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G1: the gene sets differ from notebook 71'
assert int(pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0).loc['units in the common support', 'Human vs mouse PT']) == len(INP['position'])
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = pd.Series(INP['segment']).str.replace('PT-', '', regex=False).to_numpy()
IS_HUMAN = np.asarray(INP['case'], bool)
COUNTS = INP['counts'].tocsr()
LIBRARY = np.asarray(INP['library'], float)                          # mt-free (the NB offset)
COVARIATES = INP['covariates']
UNIT_ROWS = np.flatnonzero(UNITS.obs_names.isin(INP['obs'].index))
assert (UNITS.obs_names[UNIT_ROWS] == INP['obs'].index).all()
GUARDS.append(f'G1: export contract; pass-1 fingerprint {FINGERPRINT}; universe ({len(GENES):,}), sets ({len(SETS):,}) and '
              f'units ({len(POSITION):,}) equal notebook 71')


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


STAT71 = pd.read_csv(T71 / 'tables' / f'gene_statistics_{DS}.csv', header=[0, 1], index_col=0)
COND = STAT71['condition'].copy()
assert COND.index.astype(str).tolist() == GENES
MODEL = stored(T71, f'gene_model_{DS}_species')                                            # G2
np.testing.assert_allclose(MODEL['stat|T_spatial'], COND.T_spatial.to_numpy(), rtol=1e-12, atol=0)
LOG_MEAN_COUNT = np.log(np.asarray(COUNTS.mean(axis=0)).ravel())
again = moderated_f(COND.T_spatial, DF1, COND.LR_spec.to_numpy(), DF2, covariate=LOG_MEAN_COUNT, robust=False)
np.testing.assert_allclose(again.z.to_numpy(), COND.z.to_numpy(), rtol=1e-9, atol=1e-9)
GRID = MODEL['grid']
np.testing.assert_allclose(GRID, common_grid(POSITION, SPECIMEN, 101), rtol=0, atol=1e-12)
Z = COND.z.astype(float)
OK71 = COND.converged.astype(bool) & ~COND.separated.astype(bool)
GUARDS.append("G2: notebook 71's cached condition model reproduces its T_spatial and moderated-F z")

TESTS71 = {part: pd.read_csv(T71 / 'tables' / f'pathway_tests_{DS}_cameraPR_{f}.csv').set_index('pathway_id').reindex(list(SETS))
           for part, f in PARTITION_FILES.items()}
CALLS = [p for p in SETS if bool(TESTS71['condition'].call.get(p, False))]
assert len(CALLS) == 83, len(CALLS)                                                         # G3
FDR = TESTS71['condition'].fdr
REDUNDANCY = pd.read_csv(T71 / 'tables' / f'redundancy_clusters_{DS}.csv', index_col=0).reindex(list(SETS))
CONTRIB = set(Z.nlargest(int(round(TOP_SHARE * len(Z)))).index)                            # notebook 75's rule
P75 = pd.read_csv(T75 / 'tables' / f'programs_{DS}.csv', index_col=0)
programs_again = overlap_programs(SETS, sorted(CALLS), contributing=CONTRIB, cut=PROGRAM_CUT).set_index('pathway_id').program
assert set(P75.index) == set(CALLS) and (programs_again.reindex(P75.index) == P75.program).all()
PROGRAM, PROGRAM_NAME = P75.program.reindex(CALLS), P75['program name'].reindex(CALLS)
PATHWAY_NAME = pd.Series({p: p.split('::', 1)[1] for p in SETS}).str.replace(r' R-HSA-\d+$', '', regex=True)
GUARDS.append(f"G3: 71's {len(CALLS)} condition calls; overlap_programs reproduces 75's {PROGRAM.nunique()} programs")

TRANSITIONS = pd.read_csv(T72 / 'tables' / 'transitions.csv')
TRANSITIONS = TRANSITIONS[TRANSITIONS.dataset.eq('hvm')].reset_index(drop=True)
again_t = logistic_transitions(POSITION, INP['segment'], SPECIMEN).merge(TRANSITIONS, on=['specimen', 'transition'])
assert len(again_t) == 8 and np.allclose(again_t.position_x, again_t.position_y, atol=1e-6)   # G4
CUTS = {'pooled': sc.mean_transitions(TRANSITIONS, NAMES), 'mouse': sc.mean_transitions(TRANSITIONS, MOUSE),
        'human': sc.mean_transitions(TRANSITIONS, HUMAN)}
GUARDS.append(f"G4: logistic transitions reproduce 72's; mean cuts {CUTS}")
if CUTS['human'] is None:
    READING.append('Human species transitions (mean of the two sections; HUK1_MED1 crosses, 72 C1) are not increasing, so '
                   "the human segment-unit mapping is undefined. Where the protocol asks for each species' own segment "
                   'units (3.3 a sensitivity, Figure 5d columns, Figure 5e x axis), the pooled transitions are used for '
                   'both species, and each species\' own label spans are marked instead.')

SEG75 = pd.read_csv(T75 / 'tables' / f'gene_statistics_segment_{DS}.csv', header=[0, 1], index_col=0)['condition']
SEGSET75 = pd.read_csv(T75 / 'tables' / f'pathway_tests_segment_{DS}.csv', header=[0, 1], index_col=0)
assert SEG75.index.astype(str).tolist() == GENES and set(SEGSET75.index) == set(SETS)        # G5
T74_FILES = {'beyond': T74 / 'tables' / 'beyond_steps_sets_P.csv',
             'spline_steps': T74 / 'tables' / f'pathway_tests_ladder_steps_on_spline_{DS}.csv'}
HAS74 = all(p.is_file() for p in T74_FILES.values())
GUARDS.append("G5: 75's segment tables cover 71's universe and sets; notebook 74's tables "
              + ('present' if HAS74 else 'absent (their columns are NaN)'))
record73 = json.loads((T73 / 'run_record.json').read_text())
assert record73['confirmed_labels']['membership_sha1'] == FINGERPRINT                         # G6
CLASSES = pd.read_csv(T73 / 'tables' / '43' / 'gene_classes_symmetric.csv', index_col=0)
CLASSES_MT = pd.read_csv(T73 / 'tables' / 'mt_in_library' / '43' / 'gene_classes_symmetric.csv', index_col=0)
GUARDS.append(f"G6: notebook 73's classes come from its run with the confirmed fingerprint {FINGERPRINT}")

panels = PROV['config']['marker_panels_yaml']
LABEL_PANEL = {str(g) for s in ('PT-S1', 'PT-S2', 'PT-S3') for g in panels['label_panels'][s]}
ANCHORS = {str(g) for side in ('early', 'late') for g in panels['axis_anchors'][side]}
INTEGRATION = harmony_features(results_root, 'hvm', 'pass1') | harmony_features(results_root, 'hvm', 'pass2')
Y_MTFREE = COUNTS.multiply((1e4 / LIBRARY)[:, None]).tocsr()
Y_MTFREE.data = np.log1p(Y_MTFREE.data)
print('\n'.join('· ' + g for g in GUARDS))
print(f'Inputs {time.perf_counter() - t0:.0f} s · {len(POSITION):,} units · {len(GENES):,} genes · {len(SETS):,} sets · '
      f'{len(CALLS)} calls in {REDUNDANCY.cluster_id.reindex(CALLS).nunique()} redundancy clusters and {PROGRAM.nunique()} programs')

## 3 · Categories (3.2; notebook 43 §7's rules unchanged)

Each call's contributing members (the top 10% of 71's condition z) are classed over S2 − S1 and S3c − early with
notebook 73's symmetric classes (primary arm, mt-free library): *shared* (conserved or reversal in some contrast),
*mouse-led*, *human-led*, *mixed*. A call with fewer than 3 zonated contributing members is *unresolved*; otherwise
a *summary of mouse zonation* (≥ 50% mouse-led), *shared*, *human-led* or *mixed*. Sensitivity: 73's secondary arm
(mt in the library). Per the reading of 72's C12, "mouse-led" rests on the label-based classes only.

In [ ]:
CATEGORY_ORDER = ['summary of mouse zonation', 'shared', 'human-led', 'mixed', 'unresolved']
CAT = sc.call_categories(SETS, CALLS, CONTRIB, CLASSES)
CAT_MT = sc.call_categories(SETS, CALLS, CONTRIB, CLASSES_MT)
CAT['category (mt in library)'] = CAT_MT.category
CAT['label-panel members'] = [len(set(SETS[p]) & CONTRIB & LABEL_PANEL) for p in CALLS]
READING.append("Categories use the contributing members only, without notebook 75's fallback to all members when fewer "
               "than 3 contribute: 3.2 says 43 §7's rules are unchanged, and 43 §7 had no fallback (such calls are "
               'unresolved). Contributing genes are the top 10% of the condition z by `nlargest` (75\'s exact rule).')


def cluster_labels(values):
    """Each redundancy cluster with a call takes the label of its lowest-FDR called member."""
    frame = pd.DataFrame({'cluster': REDUNDANCY.cluster_id.reindex(CALLS), 'fdr': FDR.reindex(CALLS), 'value': values.reindex(CALLS)})
    return frame.sort_values(['fdr']).groupby('cluster').value.first()


CATEGORY_COUNTS = pd.DataFrame({
    'calls': CAT.category.value_counts(), 'calls (mt in library)': CAT['category (mt in library)'].value_counts(),
    'redundancy clusters': cluster_labels(CAT.category).value_counts(),
    'programs (calls)': pd.Series([CAT.category[PROGRAM.eq(k)].mode().iloc[0] for k in PROGRAM.unique()]).value_counts()
}).reindex(CATEGORY_ORDER).fillna(0).astype(int)
CATEGORY_COUNTS.loc['total'] = CATEGORY_COUNTS.sum()
CAT.join(PATHWAY_NAME.rename('name')).to_csv(tables / 'call_categories.csv')
CATEGORY_COUNTS.to_csv(tables / 'category_counts.csv')
display(CATEGORY_COUNTS)
resolved = CAT.category.ne('unresolved')
print(f"Resolved calls: {int(resolved.sum())}; summary of mouse zonation {int(CAT.category.eq('summary of mouse zonation').sum())} "
      f"({CAT.category.eq('summary of mouse zonation').sum() / max(resolved.sum(), 1):.2f} of resolved); human-led "
      f"{int(CAT.category.eq('human-led').sum())}; calls with a label-panel contributing member: {int((CAT['label-panel members'] > 0).sum())}")
READING.append("Programs take the most common category of their calls (ties: the first in pandas' mode order) for the "
               "programs column of category_counts.csv.")

## 4 · Location and direction (3.3)

**(b) Primary, coordinate-free (O6).** z_S = Δ_S / SE_S per gene and segment from notebook 75's condition steps fit.
On the three segments, `local_pathway_curves` gives each call's rank divergence D(S) against all genes; the peak
segment is its argmax, and the direction the sign of the median member z_S there (> 0: human-high).
**(a) Sensitivity, the coordinate (47 (e) unchanged).** z(x) = Δ(x) / SE(x) from 71's cache; the peak of D(x) is
converted to segment units with the pooled transitions (and with each species' where defined). Genes that are
non-converged, separated or non-finite leave the ranking. Only the segment is read (C14).

In [ ]:
ok_seg = SEG75.converged.astype(bool) & ~SEG75.separated.astype(bool)
ZS = pd.DataFrame({k: SEG75[f'delta_{s}'] / SEG75[f'se_{s}'] for k, s in enumerate(SEGMENTS)}, index=GENES).astype(float)
ZS[~ok_seg.to_numpy()] = np.nan
LOC_B = sc.call_peaks(ZS, SETS, CALLS, np.array([0., 1., 2.]))
ZX = pd.DataFrame(MODEL['delta'] / MODEL['se'], index=GENES)
ZX[~OK71.to_numpy()] = np.nan
LOC_A = sc.call_peaks(ZX, SETS, CALLS, GRID)
SEG_NAME = dict(enumerate(SEGMENTS))
LOC = pd.DataFrame({'name': PATHWAY_NAME.reindex(CALLS), 'program': PROGRAM, 'fdr': FDR.reindex(CALLS),
                    'segment (b)': LOC_B.peak_position.astype(int).map(SEG_NAME),
                    'direction (b)': np.where(LOC_B.human_high, 'human-high', 'mouse-high'),
                    'median member z_S at peak (b)': LOC_B.median_member_z_at_peak, 'members used (b)': LOC_B.members_used,
                    'peak position (a)': LOC_A.peak_position,
                    'direction (a)': np.where(LOC_A.human_high, 'human-high', 'mouse-high')}, index=CALLS)
for label, cuts in CUTS.items():
    units = sc.to_segment_units(LOC_A.peak_position, cuts)
    LOC[f'segment units (a, {label})'] = units
    LOC[f'segment (a, {label})'] = pd.Series(sc.segment_index(units), index=CALLS).map(SEG_NAME)
LOC['segments agree (b, a pooled)'] = LOC['segment (b)'].eq(LOC['segment (a, pooled)'])
LOC['directions agree (b, a)'] = LOC['direction (b)'].eq(LOC['direction (a)'])
LOC.to_csv(tables / 'call_locations.csv')
seg_index = LOC_B.peak_position.astype(int)
LOCATION = {
    'calls': len(CALLS), 'segments agree (b vs a pooled)': int(LOC['segments agree (b, a pooled)'].sum()),
    'segments agree (b vs a mouse transitions)': int(LOC['segment (b)'].eq(LOC['segment (a, mouse)']).sum()),
    'directions agree (b vs a)': int(LOC['directions agree (b, a)'].sum()),
    'human-high (b)': int(LOC['direction (b)'].eq('human-high').sum()), 'mouse-high (b)': int(LOC['direction (b)'].eq('mouse-high').sum()),
    'median segment index, human-high (b)': float(seg_index[LOC_B.human_high].median()) if LOC_B.human_high.any() else np.nan,
    'median segment index, mouse-high (b)': float(seg_index[~LOC_B.human_high].median()) if (~LOC_B.human_high).any() else np.nan,
    'median segment units (a pooled), human-high (a) — within-segment fraction not interpreted (C14)':
        float(LOC.loc[LOC['direction (a)'].eq('human-high'), 'segment units (a, pooled)'].median()),
    'median segment units (a pooled), mouse-high (a) — within-segment fraction not interpreted (C14)':
        float(LOC.loc[LOC['direction (a)'].eq('mouse-high'), 'segment units (a, pooled)'].median())}
SEG_BY_DIR = pd.crosstab(LOC['segment (b)'], LOC['direction (b)']).reindex(index=list(SEGMENTS), fill_value=0)
SEG_BY_DIR_CLUSTERS = pd.crosstab(cluster_labels(LOC['segment (b)']), cluster_labels(LOC['direction (b)'])).reindex(
    index=list(SEGMENTS), fill_value=0)
SEG_BY_DIR_A = pd.crosstab(LOC['segment (a, pooled)'], LOC['direction (a)']).reindex(index=list(SEGMENTS), fill_value=0)
location_summary = pd.concat({'calls (b)': SEG_BY_DIR, 'redundancy clusters (b)': SEG_BY_DIR_CLUSTERS,
                              'calls (a pooled)': SEG_BY_DIR_A}, axis=1).fillna(0).astype(int)
location_summary.to_csv(tables / 'location_summary.csv')
pd.Series(LOCATION).to_csv(tables / 'location_numbers.csv', header=['value'])
display(location_summary)
display(pd.Series(LOCATION).to_frame('value'))

## 5 · Coordinate-free support and robustness (3.4; no refit, O1)

Per call: 71's relabeling FDRs; notebook 75's segment F-test (coordinate-free) and its joint-test sensitivity;
notebook 74's steps on the spline and test A/B (beyond label / coordinate steps); 72's count-split arms a and c; 71's
four winsor caps; and five gene-removal checks with notebook 37's rule (cameraPR on 71's condition z restricted to
the subset, sets with 10–300 remaining members; retained = Up, FDR ≤ 0.10 and effect ratio ≥ 0.7). Core = called
and retained under detected-in-both and sex-unbiased; probe-sensitive = not retained on equal-probe genes.
Coordinate and registration robustness are notebook 72's (C8 and C14 not met).

In [ ]:
t0 = time.perf_counter()
ROB = pd.DataFrame(index=pd.Index(CALLS, name='pathway_id'))
ROB['name'] = PATHWAY_NAME.reindex(CALLS)
for part in ('relabeling 1', 'relabeling 2'):
    ROB[f'FDR {part} (71)'] = TESTS71[part].fdr.reindex(CALLS)
    ROB[f'called {part} (71)'] = TESTS71[part].call.reindex(CALLS).astype(bool)
ROB['segment F-test (75)'] = SEGSET75[('condition', 'call')].reindex(CALLS).astype(bool)
ROB['segment F-test FDR (75)'] = SEGSET75[('condition', 'fdr')].reindex(CALLS)
seg_joint = pd.read_csv(T75 / 'tables' / f'pathway_tests_segment_joint_{DS}.csv', header=[0, 1], index_col=0)
ROB['segment steps, NB joint test (75)'] = seg_joint[('condition', 'call')].reindex(CALLS).astype(bool)
if HAS74:
    beyond = pd.read_csv(T74_FILES['beyond'], header=[0, 1], index_col=0)
    spline = pd.read_csv(T74_FILES['spline_steps'], header=[0, 1], index_col=0)
    ROB['steps on the spline (74)'] = spline[('condition', 'call')].reindex(CALLS).astype(bool)
    ROB['beyond label steps, test A (74)'] = beyond[('A · condition', 'call')].reindex(CALLS).astype(bool)
    ROB['beyond coordinate steps, test B (74)'] = beyond[('B · condition', 'call')].reindex(CALLS).astype(bool)
else:
    for column in ('steps on the spline (74)', 'beyond label steps, test A (74)', 'beyond coordinate steps, test B (74)'):
        ROB[column] = np.nan
for arm in ('a', 'c'):
    g6 = stored(T72, f'g6_{arm}')
    called = set(g6['set_ids'].astype(str)[g6['set_call'].astype(bool)])
    ROB[f'count split arm {arm} (72)'] = [p in called for p in CALLS]
winsor = pd.read_csv(T71 / 'tables' / 's9_winsor_lost_new.csv')
lost = set(winsor.loc[winsor.dataset.eq('Human vs mouse PT') & winsor.status.eq('lost'), 'pathway_id'])
ROB['kept under every winsor cap (71)'] = [p not in lost for p in CALLS]

probes = pd.read_csv(nb38 / 'probe_counts.csv').drop_duplicates('mouse_symbol').set_index('mouse_symbol')
sex_parts = []
for seg in SEGMENTS:
    xiong = pd.read_csv(external / 'xiong2023_sex_bias' / f'PT-{seg}_fWTvsmWT.csv').drop_duplicates('symbol').set_index('symbol')
    sex_parts.append(xiong.p_adj.le(.05) & xiong.log2FC.abs().ge(1))
sex_biased = pd.concat(sex_parts, axis=1).reindex(GENES).fillna(False).any(axis=1)
probe_view = probes.reindex(GENES)
SUBSETS = {
    'detected in both (≥ 5% each species)': [g for g in GENES if COVARIATES.loc[g, 'detection_mouse'] >= DETECTION_BOTH
                                              and COVARIATES.loc[g, 'detection_human'] >= DETECTION_BOTH],
    'mouse sex-unbiased': [g for g in GENES if not sex_biased[g]],
    'flagged genes removed': [g for g in GENES if g not in set(FLAGGED_GENES)],
    'equal probe counts': [g for g in GENES if pd.notna(probe_view.loc[g, 'mouse_probes'])
                           and probe_view.loc[g, 'mouse_probes'] == probe_view.loc[g, 'human_probes']],
    'proteasome subunits removed': [g for g in GENES if g not in set(PROTEASOME)]}
RETENTION = {}
for label, keep in SUBSETS.items():
    RETENTION[label] = sc.subset_retention(Z, SETS, CALLS, keep, camera=camera_pr, q=CHECK_Q, ratio=CHECK_RATIO)
    ROB[f'retained: {label}'] = RETENTION[label].retained
    ROB[f'effect ratio: {label}'] = RETENTION[label].effect_ratio
ROB['core'] = ROB['retained: detected in both (≥ 5% each species)'] & ROB['retained: mouse sex-unbiased']
ROB['probe-sensitive'] = ~ROB['retained: equal probe counts']
CORE = [p for p in CALLS if ROB.loc[p, 'core']]
ROB.to_csv(tables / 'call_robustness.csv')
bool_columns = [c for c in ROB.columns if ROB[c].dtype == bool]
ROBUSTNESS = pd.DataFrame({'calls': ROB[bool_columns].sum().astype(int),
                           'redundancy clusters': pd.Series({c: int(cluster_labels(ROB[c].astype(float)).sum()) for c in bool_columns})})
ROBUSTNESS.loc['genes per subset'] = np.nan
for label, keep in SUBSETS.items():
    ROBUSTNESS.loc[f'genes kept: {label}', 'calls'] = len(keep)
ROBUSTNESS.to_csv(tables / 'robustness_summary.csv')
display(ROBUSTNESS)
misses = ROB.index[~ROB['segment F-test (75)']]
print(f'Calls without segment F-test support: {len(misses)}; of them probe-sensitive: {int(ROB.loc[misses, "probe-sensitive"].sum())}')
print(f'Robustness {time.perf_counter() - t0:.0f} s')
READING.append("The gene-removal checks recompute cameraPR's FDR over the subset's tested sets, with no squeezeVar refit "
               "(71's proteasome rule); the effect is the members' mean z minus that of every other subset gene.")
READING.append("Winsorizing: a call is kept when 71's s9_winsor_lost_new.csv lists it as lost under none of the four caps.")

## 6 · External replication (3.5; notebooks 48 A, 50 and 37 E3)

**Ours** (coordinate-free): per specimen, S3 − early (S1 + S2) of the mean per-unit log1p CP10k (mt-free library) by
package labels, median-centred over informative genes; interaction = human − mouse. **References**: Census human
cortex and Lake healthy cortex against Census male mice (primary) and female mice (sensitivity). **Per-set score**:
the joint replication z (exact matched moments ÷ √VIF from donor residuals). **Collections**: the calls, the core
calls, the call programs (75), the noise calls (O3: sets called under either relabeling by 75's cameraPR on
rank-normal raw T_spatial, minus the 83; 71 itself yields only 1 relabeled call) and their programs. **Nulls**: 9,999
overlap-preserving within-strata gene relabelings (matched random sets), plus the selection-matched (strata × z
quintile) and slope-free versions. **Decision** (Census male, signed, matched, calls): specific / better than random
only / not replicated.

In [ ]:
t0 = time.perf_counter()
OURS = sc.our_contrast(Y_MTFREE, GENES, SPECIMEN, SEGMENT, NAMES)
human_table = pd.read_csv(CENSUS / 'human_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
human_meta = pd.read_csv(CENSUS / 'human_pt_segment_pseudobulk_meta.csv', index_col=0)
mouse_table = pd.read_csv(CENSUS / 'mouse_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
mouse_meta = pd.read_csv(CENSUS / 'mouse_pt_segment_pseudobulk_meta.csv', index_col=0)
h_late, h_early, h_whole, _ = sc.census_donors(
    human_table, human_meta, {'kidney proximal convoluted tubule epithelial cell': 'early',
                              'epithelial cell of proximal tubule segment 3': 'S3'}, ['early'], GENES, min_cells=MIN_CELLS)
m_late, m_early, m_whole, m_info = sc.census_donors(
    mouse_table, mouse_meta, {'epithelial cell of proximal tubule segment 1': 'S1', 'epithelial cell of proximal tubule segment 2': 'S2',
                              'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'], GENES, min_cells=MIN_CELLS)
MALES, FEMALES = m_info.index[m_info.sex.eq('male')], m_info.index[m_info.sex.eq('female')]

lake = stored(T75, 'lake_cortex_pt')        # notebook 75's (= 68's) all-gene Lake cortex pseudobulk
import anndata as ad  # noqa: E402
native = ad.read_h5ad(NATIVE_COR1, backed='r')
human_ids = native.var['gene_ids'].astype(str)
native.file.close()
by_id = dict(zip(orthologs.human_symbol.map(human_ids[~human_ids.index.duplicated()]), orthologs.mouse_symbol))
by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(lake['var_ids'].astype(str), lake['var_names'].astype(str))])
mapped = target != ''
LAKE_ON_GENES = (pd.DataFrame(lake['sums'][:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T
                 .reindex(columns=GENES, fill_value=0.0))
old_lake = stored(OLD['50'], 'lake_pseudobulk')                                            # G7
old_genes = pd.read_csv(OLD['12'] / 'gene_statistics.csv', index_col=0).index.astype(str)
assert (old_lake['groups'].astype(str) == lake['groups'].astype(str)).all()
shared_genes = [g for g in GENES if g in set(old_genes)]
old_frame = pd.DataFrame(old_lake['sums'], columns=old_genes)
np.testing.assert_allclose(LAKE_ON_GENES[shared_genes].to_numpy(), old_frame[shared_genes].to_numpy(), rtol=0, atol=1e-9)
GUARDS.append(f"G7: the Lake pseudobulk on 71's universe equals notebook 50's on the {len(shared_genes):,} genes both share")
l_late, l_early, l_whole = sc.lake_donors(LAKE_ON_GENES.to_numpy(), lake['totals'], lake['groups'].astype(str), GENES,
                                          min_nuclei=MIN_CELLS, nuclei=lake['nuclei'])
REFERENCES = {('Census', 'male'): (h_late, h_early, MALES), ('Lake', 'male'): (l_late, l_early, MALES),
              ('Census', 'female'): (h_late, h_early, FEMALES), ('Lake', 'female'): (l_late, l_early, FEMALES)}
REFS = {key: sc.replication_inputs(OURS, HUMAN, MOUSE, COVARIATES, SETS, hl, he, m_late, m_early, donors)
        for key, (hl, he, donors) in REFERENCES.items()}
informative = ', '.join(f'{k[0]} {k[1]} {len(r["genes"]):,}' for k, r in REFS.items())
print(f'Donors: Census human {len(h_late)}, Lake {len(l_late)}, mouse {len(MALES)} male / {len(FEMALES)} female; '
      f'informative genes {informative}')

raw75 = pd.read_csv(T75 / 'tables' / f'pathway_tests_raw_tspatial_{DS}.csv', header=[0, 1], index_col=0)
noise_called = (raw75[('raw_camerapr · relabeling 1', 'call')].astype(bool) | raw75[('raw_camerapr · relabeling 2', 'call')].astype(bool))
NOISE = [p for p in SETS if bool(noise_called.get(p, False)) and p not in set(CALLS)]
NOISE_PROGRAMS = overlap_programs(SETS, sorted(NOISE), contributing=CONTRIB, cut=PROGRAM_CUT).set_index('pathway_id').program
PROGRAM_SETS = {'calls': PROGRAM, 'noise': NOISE_PROGRAMS}
print(f'Noise-call comparator: {len(NOISE)} sets ({int(noise_called.sum())} relabeling calls of raw-T_spatial cameraPR, '
      f'minus the calls) in {NOISE_PROGRAMS.nunique()} programs; core calls {len(CORE)}')
code_key = digest([NOTEBOOK_LOGIC_VERSION, Path(sc.__file__), Path(set_collection_null.__file__)])
RUN_SPECS = [(('Census', 'male'), 'signed', 'matched'), (('Census', 'male'), 'signed', 'selection-matched'),
             (('Census', 'male'), 'slope-free', 'matched'), (('Lake', 'male'), 'signed', 'matched'),
             (('Lake', 'male'), 'signed', 'selection-matched'), (('Lake', 'male'), 'slope-free', 'matched'),
             (('Census', 'female'), 'signed', 'matched'), (('Lake', 'female'), 'signed', 'matched')]
RUNS, ROWS = {}, []
for key, statistic, null in RUN_SPECS:
    ref = REFS[key]
    sets_ = sc.scored_sets(ref, set(CALLS) | set(NOISE), PROGRAM_SETS)
    keep_ = lambda ids: [p for p in ids if p in sets_]  # noqa: E731
    collections = {'calls': keep_(CALLS), 'core calls': keep_(CORE), 'noise calls': keep_(NOISE),
                   'calls programs': [p for p in sets_ if p.startswith('program:calls:')],
                   'noise programs': [p for p in sets_ if p.startswith('program:noise:')]}
    strata = sc.selection_strata(ref, Z) if null == 'selection-matched' else ref['strata']
    run = cached_payload(f'collection_null_{key[0]}_{key[1]}_{statistic}_{null}',
                         lambda ref=ref, statistic=statistic, sets_=sets_, collections=collections, strata=strata:
                         sc.collection_run(ref, statistic, sets_, collections, strata, n_perm=N_PERM, seed=SEED),
                         root=cache, params={'n_perm': N_PERM, 'seed': SEED, 'reference': list(key), 'statistic': statistic, 'null': null},
                         inputs={'statistic': digest(ref[statistic]), 'strata': strata, 'sets': digest(sorted(sets_.items())),
                                 'collections': digest(sorted(collections.items()))}, code=code_key)
    RUNS[(key, statistic, null)] = run
    row = sc.collection_summary(run, ['calls', 'core calls'], ['noise calls'], ('calls', 'noise'), seed=SEED)
    row = {'reference': f'{key[0]} cortex − {key[1]} mouse', 'statistic': statistic, 'null': null,
           **{f'n {name}': len(ids) for name, ids in collections.items()}, **row,
           'decision (calls)': sc.replication_decision(row, 'calls', 'noise calls'),
           'decision (core calls)': sc.replication_decision(row, 'core calls', 'noise calls')}
    ROWS.append(row)
REPLICATION = pd.DataFrame(ROWS)
REPLICATION.to_csv(tables / 'replication_overlap_null.csv', index=False)
display(REPLICATION.set_index(['reference', 'statistic', 'null']).T)
DECISIONS = REPLICATION[REPLICATION.statistic.eq('signed') & REPLICATION.null.eq('matched')].set_index('reference')[
    ['decision (calls)', 'decision (core calls)']]
DECISIONS.to_csv(tables / 'replication_decision.csv')
display(DECISIONS)

PER_CALL, PER_CALL_COUNTS = {}, {}
for key in REFS:
    run = RUNS[(key, 'signed', 'matched')]
    out = sc.per_call_replication(pd.Series(run['set_z'], index=run['set_ids'].astype(str)), CALLS)
    PER_CALL[key] = out
    PER_CALL_COUNTS[f'{key[0]} cortex − {key[1]} mouse'] = {'calls testable': len(out), 'replicated (BH ≤ 0.10 within list)': int(out.replicated.sum()),
                                                            'core replicated': int(out.replicated.reindex(CORE).fillna(False).sum())}
PER_CALL_COUNTS = pd.DataFrame(PER_CALL_COUNTS)
pd.concat({f'{k[0]} {k[1]}': v for k, v in PER_CALL.items()}, axis=1).to_csv(tables / 'replication_per_call.csv')
display(PER_CALL_COUNTS)
T_LEVEL = pd.Series(MODEL['stat|T_level'], index=GENES)
GENE_LEVEL = pd.DataFrame({f'{k[0]} cortex − {k[1]} mouse': sc.gene_agreement(
    ref, Z, {'mean expression': COVARIATES.mean_expression, 'T_level (NB)': T_LEVEL}, seed=GENE_SEED) for k, ref in REFS.items()})
GENE_LEVEL.to_csv(tables / 'replication_genes.csv')
display(GENE_LEVEL.round(3))
COMPONENT = pd.DataFrame({'ours (package units, labels)': REFS[('Census', 'male')]['ours_component'],
                          **{f'{k[0]} cortex − {k[1]} mouse': r['component'] for k, r in REFS.items()}})
COMPONENT.to_csv(tables / 'global_component.csv')
display(COMPONENT.round(3))
print(f'Replication {time.perf_counter() - t0:.0f} s')
READING.append('Per-call replication (37 E3) uses the joint z with exact matched moments from the collection run '
               "(50's scorer), not notebook 37's Monte Carlo matched test; the one-sided p is Φ(−z).")
READING.append("Gene-level agreement: the score is 71's condition z (K2); the comparison scores are the top 10% by mean "
               "expression and by NB T_level from 71's cache (37's T_level row).")
READING.append('Noise-call programs use the same program rule as the call programs (75: sorted by pathway id, contributing = '
               'top 10% of z).')

## 7 · Conventional-only sets (3.6; notebook 50 part 3, secondary, kept by O4)

Conventional calls are notebook 75's whole-PT or S1/S2/S3 DESeq2 with signed GSEA (condition); conventional-only means
not among the 83. Test unchanged: per-donor whole-PT log2 CPM, median-centred, Welch t of human against male mouse
donors signed by 75's whole-PT log2 fold change, joint z, the overlap-preserving collection null and D under joint
relabeling, beside the overlap-naive Mann–Whitney.

In [ ]:
from scipy.stats import mannwhitneyu  # noqa: E402
from pseudospace.set_collection_null import JointReplicationZ, collection_null, upper_p  # noqa: E402

gw = pd.read_csv(T75 / 'tables' / f'pathway_tests_deseq2_wholept_gsea_{DS}.csv', header=[0, 1], index_col=0)[('condition', 'call')]
gs = pd.read_csv(T75 / 'tables' / f'pathway_tests_deseq2_segments_gsea_{DS}.csv', header=[0, 1], index_col=0)[('condition', 'call')]
CONVENTIONAL = set(gw.index[gw.astype(bool)]) | set(gs.index[gs.astype(bool)])
de75 = pd.read_csv(T75 / 'tables' / f'deseq2_genes_{DS}.csv')
WHOLE_LFC = de75[de75.partition.eq('species') & de75.scope.eq('whole_PT')].set_index('gene').log2FoldChange
CONV_ROWS = {}
for key, human_whole in ((('Census', 'male'), h_whole), (('Lake', 'male'), l_whole)):
    avg = sc.average_level_inputs(REFS[key]['genes'], SETS, COVARIATES, human_whole, m_whole, MALES, WHOLE_LFC)
    conv_only = sorted((CONVENTIONAL - set(CALLS)) & set(avg['sets']))
    others = [p for p in avg['sets'] if p not in set(conv_only)]

    def conventional_null(avg=avg, conv_only=conv_only, others=others):
        scorer = JointReplicationZ(avg['statistic'], avg['strata'], avg['sets'], avg['residuals'])
        observed, draws = collection_null(scorer, {'conventional-only': conv_only, 'other': others}, avg['strata'], N_PERM, SEED)
        return {'observed': observed.to_numpy(), 'draws': draws.to_numpy(), 'set_ids': np.array(scorer.ids, dtype=str),
                'set_z': scorer.z()}

    run = cached_payload(f'conventional_only_null_{key[0]}', conventional_null, root=cache,
                         params={'n_perm': N_PERM, 'seed': SEED},
                         inputs={'statistic': digest(avg['statistic']), 'strata': avg['strata'], 'sets': digest(sorted(avg['sets'].items())),
                                 'conventional': sorted(conv_only)}, code=code_key)
    z = pd.Series(run['set_z'], index=run['set_ids'].astype(str))
    draws = pd.DataFrame(run['draws'], columns=['conventional-only', 'other'])
    observed = pd.Series(run['observed'], index=['conventional-only', 'other'])
    CONV_ROWS[f'{key[0]} cortex − male mouse'] = {
        'conventional calls (75)': len(CONVENTIONAL), 'conventional-only sets tested': len(conv_only),
        'median z, conventional-only': float(z.loc[conv_only].median()), 'median z, other': float(z.loc[others].median()),
        'Mann–Whitney p (overlap-naive, as notebook 37)': mannwhitneyu(z.loc[conv_only], z.loc[others], alternative='greater').pvalue,
        'mean z, conventional-only': observed['conventional-only'], 'mean z, other': observed['other'],
        'p, conventional-only vs random collections': upper_p(observed['conventional-only'], draws['conventional-only']),
        'D = mean(conventional-only) − mean(other)': observed['conventional-only'] - observed['other'],
        'p, D under joint relabeling': upper_p(observed['conventional-only'] - observed['other'], draws['conventional-only'] - draws['other'])}
CONVENTIONAL_TABLE = pd.DataFrame(CONV_ROWS)
CONVENTIONAL_TABLE.to_csv(tables / 'conventional_only_overlap_null.csv')
CONVENTIONAL_DECISION = ('group claim stands with overlap respected'
                         if CONVENTIONAL_TABLE.loc['p, D under joint relabeling', 'Census cortex − male mouse'] <= .05
                         else 'group claim not supported once overlap is respected')
display(CONVENTIONAL_TABLE)
print('Decision (Census, primary):', CONVENTIONAL_DECISION)

## 8 · Lead genes (3.7)

Genes: notebook 41's nine plus Rbp4, fixed before the run. Per gene: the percentile of 71's condition z (and of the NB
T_spatial), q, the calls in which it contributes; notebook 38's "ours" S3 − S1 per specimen on 71's units and package
labels (mt-free library; full library as the sensitivity); 38's `ours` and `ours_attenuation` tests rerun, with every
external test read from 38's saved evidence; gene and story verdicts; 73's union class; the label-panel, axis-anchor
and integration-feature flags. Program drivers follow notebook 37's rule on 75's programs.

In [ ]:
from pseudospace.replication_rules import arm_passes, gene_verdict, story_verdict  # noqa: E402

EVID = pd.read_csv(nb38 / 'story_gene_evidence.csv')
claim_genes = list(dict.fromkeys(list(EVID.gene) + LEAD_GENES))
var = pd.Index(UNITS.var_names.astype(str))
present = [g for g in claim_genes if g in var]
X_claim = UNITS.X[UNIT_ROWS][:, var.get_indexer(present)]
X_claim = sparse.csr_matrix(X_claim, dtype=float)


def s3_minus_s1(library):
    y = X_claim.multiply((1e4 / library)[:, None]).tocsr()
    y.data = np.log1p(y.data)
    out = {}
    for name in NAMES:
        rows = SPECIMEN == name
        out[name] = (np.asarray(y[rows & (SEGMENT == 'S3')].mean(axis=0)).ravel()
                     - np.asarray(y[rows & (SEGMENT == 'S1')].mean(axis=0)).ravel())
    return pd.DataFrame(out, index=present)


OURS31 = s3_minus_s1(LIBRARY)
OURS31_FULL = s3_minus_s1(np.asarray(INP['library_with_excluded'], float))
DETECT = pd.DataFrame({'detection_mouse': np.asarray((X_claim[~IS_HUMAN] > 0).mean(axis=0)).ravel(),
                       'detection_human': np.asarray((X_claim[IS_HUMAN] > 0).mean(axis=0)).ravel()}, index=present)
# Attenuation a (38's rule): control genes chosen by external data, over 71's universe.
y_all = Y_MTFREE
ours_all = {}
for name in NAMES:
    rows = SPECIMEN == name
    ours_all[name] = (np.asarray(y_all[rows & (SEGMENT == 'S3')].mean(axis=0)).ravel()
                      - np.asarray(y_all[rows & (SEGMENT == 'S1')].mean(axis=0)).ravel())
ours_all = pd.DataFrame(ours_all, index=GENES)
oh_all, om_all = ours_all[HUMAN].mean(axis=1), ours_all[MOUSE].mean(axis=1)
lake_contrasts = pd.read_csv(nb38 / 'gene_level_lake_contrasts.csv', index_col=0).reindex(GENES)
lake_h, sn_m = lake_contrasts.lake_human_S3_minus_S1, lake_contrasts.mouse_sn_male_S3_minus_S1
balanced = set(probes.index[probes.probe_balance.eq('balanced')])
controls = (COVARIATES[['detection_mouse', 'detection_human']].min(axis=1).ge(.10) & pd.Index(GENES).isin(balanced)
            & lake_contrasts.lake_max_cpm.ge(10) & lake_h.abs().ge(1) & sn_m.abs().ge(1) & (np.sign(lake_h) == np.sign(sn_m)))
ATTENUATION = float((oh_all / lake_h)[controls].median() / (om_all / sn_m)[controls].median())
rng = np.random.default_rng(38)
idx = np.flatnonzero(controls.to_numpy())
boot = [np.median((oh_all / lake_h).iloc[pick]) / np.median((om_all / sn_m).iloc[pick])
        for pick in (rng.choice(idx, size=len(idx), replace=True) for _ in range(2000))]
print(f'Attenuation a = {ATTENUATION:.3f} (95% {np.quantile(boot, .025):.2f}–{np.quantile(boot, .975):.2f}; '
      f'{int(controls.sum())} controls; old 0.859 on 70)')
READING.append("38's attenuation controls are chosen with the external contrasts of 38's gene_level_lake_contrasts.csv "
               "(old universe), so genes new to 71's universe cannot be controls; our contrasts use the mt-free library.")


def claim_of(r):
    get = lambda c: '' if pd.isna(r[c]) else str(r[c])  # noqa: E731
    return {'mouse': get('claim_mouse'), 'human': get('claim_human'), 'interaction': get('claim_interaction')}


def tests_for(r, ours31, detect):
    claim, one = claim_of(r), ('' if pd.isna(r.one_species) else r.one_species)
    g = r.gene
    keys = ['ours'] + (['ours_attenuation'] if claim['human'] == '0' and ATTENUATION < 1 else [])
    keys += ['human_lake'] if claim['human'] != '' else []
    keys += ['int_vs_male_sn', 'int_vs_female_sn'] if claim['interaction'] != '' and one == '' else []
    keys += ['mouse_sn_male', 'mouse_sn_female', 'mouse_micro_male', 'mouse_micro_female'] if claim['mouse'] != '' and one != 'human' else []
    keys += (['mouse_absent'] if one == 'human' else []) + (['human_absent'] if one == 'mouse' else [])
    saved = {k: (None if pd.isna(r[f'test_{k}']) else bool(r[f'test_{k}'])) for k in keys}
    if g in ours31.index:
        values = ours31.loc[g]
        saved['ours'] = sc.ours_test(claim, values, HUMAN, MOUSE)
        if 'ours_attenuation' in keys:
            saved['ours_attenuation'] = arm_passes('0', values[HUMAN].mean() / ATTENUATION, values[MOUSE].mean())
        detection_ok = bool(detect.loc[g].min() >= .10) or one != ''
    else:
        saved['ours'], detection_ok = None, one != ''
        if 'ours_attenuation' in keys:
            saved['ours_attenuation'] = None
    return saved, detection_ok


VERDICTS, VERDICTS_FULL, TEST_ROWS = {}, {}, []
for _, r in EVID.iterrows():
    probe_ok = bool(probes.loc[r.gene, 'mouse_probes'] == probes.loc[r.gene, 'human_probes']) if r.gene in probes.index else False
    for ours31, store in ((OURS31, VERDICTS), (OURS31_FULL, VERDICTS_FULL)):
        tests, detection_ok = tests_for(r, ours31, DETECT)
        verdict = gene_verdict(tests, critical={'ours', 'human_lake', 'mouse_absent', 'human_absent'},
                               probe_balanced=probe_ok, detection_ok=detection_ok)
        store[r.gene] = 'example (species claim dropped a priori)' if r.role == 'example' else verdict
        if store is VERDICTS:
            TEST_ROWS.append({'gene': r.gene, 'story': r.story, 'role': r.role, **{f'test_{k}': v for k, v in tests.items()},
                              'verdict': store[r.gene], 'old verdict': r.verdict, 'old test_ours': r.test_ours})
GENE_TESTS = pd.DataFrame(TEST_ROWS).set_index('gene')
OLD_STORIES = pd.read_csv(nb38 / 'story_verdicts.csv').set_index('story')
story_rows = []
for story in STORY_ORDER:
    block = EVID[EVID.story.eq(story)].set_index('gene')
    probe_bal = pd.Series({g: bool(probes.loc[g, 'mouse_probes'] == probes.loc[g, 'human_probes']) for g in block.index})
    if story == 'probe_example':
        story_rows.append({'story': story, 'verdict': 'example', 'old verdict': 'example'})
        continue
    leads = block.index[block.role.eq('lead')].tolist()
    out = {}
    for label, store in (('verdict', VERDICTS), ('verdict (full library)', VERDICTS_FULL)):
        v, share = story_verdict(pd.Series({g: store[g] for g in block.index}), leads, probe_bal, can_lead=story in LEAD_STORIES)
        out[label], out[f'share ({label})'] = v, share
    story_rows.append({'story': story, **out, 'leads': ', '.join(f'{g} ({VERDICTS[g]})' for g in leads),
                       'old verdict': OLD_STORIES.loc[story, 'verdict']})
STORIES = pd.DataFrame(story_rows).set_index('story')
STORIES.to_csv(tables / 'lead_stories.csv')
display(STORIES)

z_pct = Z.rank(ascending=False, pct=True)
t_pct = COND.T_spatial.rank(ascending=False, pct=True)
old37 = pd.read_csv(OLD['37'] / 'tables' / 'gene_level_statistics.csv', index_col=0).T_spatial
old_pct = old37.rank(ascending=False, pct=True)
old_classes = pd.read_csv(OLD['43'] / 'tables' / 'gene_classes_symmetric.csv', index_col=0)['union S2-S1 + S3c-early']
evid_by_gene = EVID.set_index('gene')
LEAD_ROWS = []
for g in LEAD_GENES:
    in_u = g in set(GENES)
    contributes = [p for p in CALLS if g in SETS[p] and g in CONTRIB]
    row = {'gene': g, 'in 71 universe': in_u, 'z percentile (top)': z_pct.get(g, np.nan), 'z': Z.get(g, np.nan),
           'q': COND.q.get(g, np.nan), 'T_spatial percentile (top)': t_pct.get(g, np.nan),
           'old T_spatial percentile (37)': old_pct.get(g, np.nan), 'calls it contributes to': len(contributes),
           'calls (names)': '; '.join(PATHWAY_NAME[p] for p in contributes[:6]),
           'class (73)': CLASSES['union S2-S1 + S3c-early'].get(g, np.nan), 'old class (43)': old_classes.get(g, np.nan),
           'label-panel gene': g in LABEL_PANEL, 'axis anchor': g in ANCHORS, 'integration feature': g in INTEGRATION,
           'probes M/H': (f"{int(probes.loc[g, 'mouse_probes'])}/{int(probes.loc[g, 'human_probes'])}" if g in probes.index else '')}
    for label, frame in (('mt-free', OURS31), ('full library', OURS31_FULL)):
        if g in frame.index:
            row[f'ours human S3 − S1 ({label})'] = float(frame.loc[g, HUMAN].mean())
            row[f'ours mouse S3 − S1 ({label})'] = float(frame.loc[g, MOUSE].mean())
            row[f'ours interaction ({label})'] = row[f'ours human S3 − S1 ({label})'] - row[f'ours mouse S3 − S1 ({label})']
    if g in evid_by_gene.index:
        e = evid_by_gene.loc[g]
        row.update({'story': e.story, 'role': e.role, 'verdict': VERDICTS[g], 'old verdict': e.verdict,
                    'old ours human S3 − S1': e.ours_human, 'old ours mouse S3 − S1': e.ours_mouse})
    else:
        row.update({'story': 'reference', 'role': 'reference', 'verdict': 'no claim', 'old verdict': 'no claim'})
    LEAD_ROWS.append(row)
LEAD = pd.DataFrame(LEAD_ROWS).set_index('gene')
LEAD.to_csv(tables / 'lead_genes.csv')
GENE_TESTS.to_csv(tables / 'lead_gene_tests.csv')
display(LEAD.T)
READING.append('The ours arm uses every export gene (shared 1:1 orthologs) on 71\'s units, so claim genes outside 71\'s '
               'universe still get values; detection for the verdict is measured on the same units.')

DRIVER_ROWS = []
for program, block in PROGRAM.groupby(PROGRAM):
    ids = list(block.index)
    signal = lambda p: ([g for g in SETS[p] if g in CONTRIB] if len([g for g in SETS[p] if g in CONTRIB]) >= 3 else list(SETS[p]))  # noqa: E731
    counts_ = pd.Series([g for p in ids for g in signal(p)]).value_counts()
    shared_ = counts_[counts_ >= max(1, len(ids) / 2)].index
    drivers = Z.reindex(shared_).sort_values(ascending=False).index[:8].tolist()
    directions = LOC.loc[ids, 'direction (b)']
    DRIVER_ROWS.append({'program': int(program), 'name': PROGRAM_NAME[ids[0]], 'calls': len(ids),
                        'direction (b; majority)': 'human-high' if directions.eq('human-high').mean() >= .5 else 'mouse-high',
                        'median peak segment index (b)': float(seg_index.reindex(ids).median()),
                        'categories': '; '.join(f'{k} {v}' for k, v in CAT.category.reindex(ids).value_counts().items()),
                        'drivers': ', '.join(drivers), 'best FDR': float(FDR.reindex(ids).min()),
                        'calls (names)': '; '.join(PATHWAY_NAME.reindex(ids))})
PROGRAM_SUMMARY = pd.DataFrame(DRIVER_ROWS).set_index('program').sort_values(['calls', 'best FDR'], ascending=[False, True])
PROGRAM_SUMMARY.to_csv(tables / 'program_summary.csv')
display(PROGRAM_SUMMARY.drop(columns='calls (names)').head(20))

## 9 · The call table (3.1) and the export for the literature pass (3.9, O5)

In [ ]:
old22 = pd.read_csv(OLD['50'] / 'tables' / 'primary_pathway_list.csv', index_col=0).index
table_s2_old = pd.read_csv(OLD['37'] / 'tables' / 'table_S2_pathways.csv', index_col=0)
old36 = table_s2_old.index[table_s2_old.robust.astype(bool)]
tests37 = pd.read_csv(OLD['37'] / 'tables' / 'matched_and_joint_tests_all_partitions.csv')
sp37 = tests37[tests37.partition.eq('species') & tests37.statistic.eq('T_spatial')]
old60 = set(sp37.loc[sp37.effect.gt(0) & sp37.q_joint.le(ALPHA), 'pathway_id'])
calls69 = pd.read_csv(OLD['69'] / 'tables' / f'pathway_tests_{DS}_cameraPR_species.csv')
old97 = set(calls69.loc[calls69.call.astype(bool), 'pathway_id'])
lit_covered = old60 | set(old36)   # the 2026-10-04 literature check covered the confident T_spatial list of 31–33
CALL_TABLE = pd.DataFrame(index=pd.Index(list(SETS), name='pathway_id'))
CALL_TABLE['name'] = PATHWAY_NAME
CALL_TABLE['library'] = [p.split('::', 1)[0] for p in SETS]
for part, frame in TESTS71.items():
    for column in ('n', 'direction', 'fdr', 'call'):
        CALL_TABLE[f'{column} · {part}'] = frame[column]
CALL_TABLE['redundancy cluster'] = REDUNDANCY.cluster_id
CALL_TABLE['program'] = PROGRAM.reindex(CALL_TABLE.index)
CALL_TABLE['program name'] = PROGRAM_NAME.reindex(CALL_TABLE.index)
CALL_TABLE = CALL_TABLE.join(CAT[['category', 'category (mt in library)', 'zonated_contributing']])
CALL_TABLE = CALL_TABLE.join(LOC[['segment (b)', 'direction (b)', 'segment (a, pooled)', 'direction (a)']])
CALL_TABLE = CALL_TABLE.join(ROB.drop(columns='name'))
for key, out in PER_CALL.items():
    CALL_TABLE[f'replication z ({key[0]}, {key[1]} mice)'] = out.z_joint
    CALL_TABLE[f'replicated ({key[0]}, {key[1]} mice)'] = out.replicated
CALL_TABLE['noise call (comparator)'] = CALL_TABLE.index.isin(NOISE)
CALL_TABLE['conventional call (75)'] = CALL_TABLE.index.isin(CONVENTIONAL)
for label, ids in (('old 22', old22), ('old 36', old36), ('old 60', old60), ("69's 97", old97)):
    CALL_TABLE[f'in {label}'] = CALL_TABLE.index.isin(list(ids))
CALL_TABLE.to_csv(tables / 'table_S2_calls.csv')

LIT_ROWS = []
for p in sorted(CALLS, key=lambda p: (FDR[p], p)):
    members = sorted([g for g in SETS[p] if g in CONTRIB], key=lambda g: -Z[g])[:8]
    LIT_ROWS.append({
        'pathway_id': p, 'name': PATHWAY_NAME[p], 'library': p.split('::', 1)[0], 'fdr (71)': FDR[p],
        'redundancy cluster': REDUNDANCY.cluster_id[p], 'program': int(PROGRAM[p]), 'program name': PROGRAM_NAME[p],
        'category': CAT.category[p], 'direction (b)': LOC.loc[p, 'direction (b)'], 'peak segment (b)': LOC.loc[p, 'segment (b)'],
        'peak segment (a, pooled)': LOC.loc[p, 'segment (a, pooled)'],
        'lead genes': '; '.join(f"{g} [{CLASSES['union S2-S1 + S3c-early'].get(g, 'unclassified')}"
                                + (', label panel' if g in LABEL_PANEL else '') + (', anchor' if g in ANCHORS else '')
                                + (', integration' if g in INTEGRATION else '') + ']' for g in members),
        'contains R6 lead genes': ', '.join(g for g in LEAD_GENES if g in SETS[p]),
        'in old 22': p in set(old22), 'in old 36': p in set(old36), 'in old confident list (literature check 2026-10-04)': p in lit_covered})
LITERATURE = pd.DataFrame(LIT_ROWS).set_index('pathway_id')
LITERATURE.to_csv(tables / 'calls_for_literature.csv')
display(LITERATURE.head(10))
READING.append("The literature export flags whether a call was in the old confident lists (37's 60 candidates or 36 robust), "
               "which the 2026-10-04 literature check covered by theme; it lists at most 8 contributing members by z per call.")

## 10 · Figures (3.8)

In [ ]:
POOLED_UNITS = sc.to_segment_units(POSITION, CUTS['pooled'])
fig, ax = plt.subplots(figsize=(88 * MM, max(60, 2.6 * PROGRAM.nunique() + 18) * MM), layout='constrained')
order = PROGRAM_SUMMARY.sort_values(['median peak segment index (b)', 'calls'], ascending=[True, False]).index
for row, program in enumerate(order):
    ids = PROGRAM.index[PROGRAM.eq(program)]
    for k, p in enumerate(ids):
        ax.scatter(seg_index[p] + (k - (len(ids) - 1) / 2) * .06, row, s=10,
                   color=HUMAN_COLOR if LOC.loc[p, 'direction (b)'] == 'human-high' else MOUSE_COLOR, lw=0)
ax.set_yticks(range(len(order)), [f"{PROGRAM_SUMMARY.loc[k, 'name'][:42]} ({PROGRAM_SUMMARY.loc[k, 'calls']})" for k in order], fontsize=4.5)
ax.set_xticks([0, 1, 2], list(SEGMENTS))
ax.set_xlim(-.6, 2.6)
ax.invert_yaxis()
ax.set_xlabel('segment where the species differ most (coordinate-free, 3.3 b)')
ax.set_title(f'Figure 5d · {len(CALLS)} positional F-test calls in {PROGRAM.nunique()} programs; '
             f'orange human-high, blue mouse-high', loc='left')
save(fig, 'fig5d_programs_by_segment')

# The heatmap program: the largest program whose direction differs from that of 75's Figure 4e program.
sizes = PROGRAM_SUMMARY[['calls', 'best FDR']]
fig4e = sizes.sort_values(['calls', 'best FDR'], ascending=[False, True]).index[0]
other = PROGRAM_SUMMARY[PROGRAM_SUMMARY['direction (b; majority)'].ne(PROGRAM_SUMMARY.loc[fig4e, 'direction (b; majority)'])]
HEAT_PROGRAM = other.sort_values(['calls', 'best FDR'], ascending=[False, True]).index[0] if len(other) else fig4e
heat_ids = sorted(PROGRAM.index[PROGRAM.eq(HEAT_PROGRAM)], key=lambda p: (FDR[p], p))
union = sorted({g for p in heat_ids for g in ([x for x in SETS[p] if x in CONTRIB] if len([x for x in SETS[p] if x in CONTRIB]) >= 3 else SETS[p])})
rows_h = list(Z.reindex(union).nlargest(MAX_ROWS).index) if len(union) > MAX_ROWS else union
n_bins = 3 * HEAT_PER_SEGMENT
heat_bin = np.clip(np.floor(POOLED_UNITS * HEAT_PER_SEGMENT), 0, n_bins - 1).astype(int)
cols_h = pd.Index(GENES).get_indexer(rows_h)
spec_means = {n: np.column_stack([np.asarray(Y_MTFREE[(SPECIMEN == n) & (heat_bin == b)][:, cols_h].mean(axis=0)).ravel()
                                  if ((SPECIMEN == n) & (heat_bin == b)).any() else np.full(len(rows_h), np.nan)
                                  for b in range(n_bins)]) for n in NAMES}


def zscore_rows(values):
    centred = values - np.nanmean(values, axis=1, keepdims=True)
    sd = np.nanstd(values, axis=1, keepdims=True)
    return np.where(sd > 1e-9, centred / np.where(sd > 1e-9, sd, 1), 0.)


group_z = {'mouse': zscore_rows(np.nanmean([spec_means[n] for n in MOUSE], axis=0)),
           'human': zscore_rows(np.nanmean([spec_means[n] for n in HUMAN], axis=0))}
order_h = pd.DataFrame({'m': np.nanargmax(group_z['mouse'], axis=1), 'h': np.nanargmax(group_z['human'], axis=1), 'g': rows_h}).sort_values(['m', 'h', 'g']).index
rows_h = [rows_h[i] for i in order_h]
group_z = {k: v[order_h] for k, v in group_z.items()}
spec_z = {n: zscore_rows(spec_means[n])[order_h] for n in NAMES}
heat_long = pd.concat([pd.DataFrame(v, index=rows_h).rename_axis('gene').reset_index().melt(id_vars='gene', var_name='bin', value_name='z').assign(column=k)
                       for k, v in {**group_z, **spec_z}.items()])
heat_long.to_csv(tables / 'fig5d_heatmap_matrix.csv', index=False)
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402
HEAT = LinearSegmentedColormap.from_list('teal_grey_brown', ['#01665E', '#5AB4AC', '#E6E6E6', '#D8B365', '#8C510A'])
fig = plt.figure(figsize=(180 * MM, (40 + len(rows_h) * 3.4) * MM))
gsp = fig.add_gridspec(2, 5, width_ratios=[12, 12, 1.2, 2.4, 2.4], height_ratios=[1, .6], wspace=.08, hspace=.15)
for col, (sp, color, names_) in enumerate((('mouse', MOUSE_COLOR, MOUSE), ('human', HUMAN_COLOR, HUMAN))):
    ax = fig.add_subplot(gsp[0, col])
    ax.imshow(np.clip(group_z[sp], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP, aspect='auto', interpolation='nearest')
    for b in (HEAT_PER_SEGMENT - .5, 2 * HEAT_PER_SEGMENT - .5):
        ax.axvline(b, color=INK, lw=.5, ls=':')
    ax.set_xticks([1.5, 5.5, 9.5], list(SEGMENTS))
    ax.set_yticks(range(len(rows_h)), rows_h if col == 0 else [], fontsize=4.5)
    ax.set_title(f'{sp} (mean of specimens, z within {sp})', loc='left', color=color)
    sub = gsp[1, col].subgridspec(1, 2, wspace=.08)
    for j, n in enumerate(names_):
        bx = fig.add_subplot(sub[0, j])
        bx.imshow(np.clip(spec_z[n], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP, aspect='auto', interpolation='nearest')
        bx.set_xticks([1.5, 5.5, 9.5], list(SEGMENTS), fontsize=4.5)
        bx.set_yticks([])
        bx.set_title(n, loc='left', fontsize=5)
cx = fig.add_subplot(gsp[0, 2])
for i, g in enumerate(rows_h):
    cx.add_patch(plt.Rectangle((0, i - .4), 1, .8, color=CLASS_COLORS.get(CLASSES['union S2-S1 + S3c-early'].get(g, ''), '#FFFFFF')))
cx.set_xlim(0, 1)
cx.set_ylim(len(rows_h) - .5, -.5)
cx.axis('off')
cx.set_title('class', fontsize=5)
tx = fig.add_subplot(gsp[0, 3])
tx.barh(range(len(rows_h)), Z.reindex(rows_h).to_numpy(), color='#1b8a5a', height=.7)
tx.set_ylim(len(rows_h) - .5, -.5)
tx.set_yticks([])
tx.set_title('z', fontsize=5)
lx = fig.add_subplot(gsp[0, 4])
lfc = WHOLE_LFC.reindex(rows_h).clip(-6, 6).fillna(0).to_numpy()
lx.barh(range(len(rows_h)), lfc, color=np.where(lfc > 0, HUMAN_COLOR, MOUSE_COLOR), height=.7)
lx.set_ylim(len(rows_h) - .5, -.5)
lx.set_yticks([])
lx.set_title('log2 H/M', fontsize=5)
fig.suptitle(f'Figure 5d · program "{PROGRAM_SUMMARY.loc[HEAT_PROGRAM, "name"]}" ({len(heat_ids)} calls, '
             f'{PROGRAM_SUMMARY.loc[HEAT_PROGRAM, "direction (b; majority)"]}); columns: {HEAT_PER_SEGMENT} bins per segment '
             'on pooled segment units; within-segment columns are display only (C14)', x=.01, ha='left', fontsize=6)
save(fig, 'fig5d_program_heatmap')
READING.append(f"Figure 4e's program (75's rule: most calls, then lowest best FDR) is "
               f"\"{PROGRAM_SUMMARY.loc[fig4e, 'name']}\" ({PROGRAM_SUMMARY.loc[fig4e, 'direction (b; majority)']}); the "
               f"Figure 5d heatmap shows \"{PROGRAM_SUMMARY.loc[HEAT_PROGRAM, 'name']}\".")

# Figure 5e · lead-gene cards.
male_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
male_bulk = male_bulk.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
micro_male = pd.DataFrame({s: male_bulk[[c for c in male_bulk.columns if f'_PT{s}_' in c]].apply(pd.to_numeric, errors='coerce')
                           .clip(lower=0).mean(axis=1) for s in SEGMENTS})
sex_bulk = pd.read_csv(external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz', sep='\t')
sex_bulk = sex_bulk.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
micro_female = sex_bulk[['fPTS1', 'fPTS2', 'fPTS3']].set_axis(list(SEGMENTS), axis=1).apply(pd.to_numeric, errors='coerce')
lake_means = pd.read_csv(nb38 / 'lake_cortex_segment_means.csv', index_col=0)
mm = mouse_meta[~mouse_meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
mm['segment'] = mm.cell_type.map({'epithelial cell of proximal tubule segment 1': 'S1', 'epithelial cell of proximal tubule segment 2': 'S2',
                                  'epithelial cell of proximal tubule segment 3': 'S3'})
mm = mm[mm.segment.notna() & mm.n_cells.ge(MIN_CELLS)]
complete = mm.groupby('donor_id').segment.apply(lambda s: {'S1', 'S2', 'S3'}.issubset(set(s)))
mm = mm[mm.donor_id.isin(complete[complete].index)]
sn_log = np.log2(mouse_table.loc[mm.index].div(mm.total_counts_all_genes, axis=0) * 1e6 + 1)
SN_MEANS = {sex: pd.DataFrame({s: sn_log.loc[mm.index[mm.sex.eq(sex) & mm.segment.eq(s)]].mean() for s in SEGMENTS})
            for sex in ('male', 'female')}


def external_profiles(g):
    out = {}
    if g in lake_means.index:
        v = lake_means.loc[g, ['lake_log2cpm_S1', 'lake_log2cpm_S2', 'lake_log2cpm_S3']].to_numpy(float)
        out['Lake human cortex'] = v - v[0]
    for sex, tpm in (('male', micro_male), ('female', micro_female)):
        if g in tpm.index:
            v = np.log2(tpm.loc[g, list(SEGMENTS)].to_numpy(float) + 1)
            if np.isfinite(v).all():
                out[f'mouse microdissection, {sex}'] = v - v[0]
    for sex in ('male', 'female'):
        if g in SN_MEANS[sex].index:
            v = SN_MEANS[sex].loc[g].to_numpy(float)
            out[f'mouse snRNA, {sex}'] = v - v[0]
    return out


STYLE = {'Lake human cortex': (HUMAN_COLOR, '-', 'o', True), 'mouse microdissection, male': (MOUSE_COLOR, '-', 's', True),
         'mouse microdissection, female': (MOUSE_COLOR, '--', 's', False), 'mouse snRNA, male': (MOUSE_COLOR, ':', 'o', True),
         'mouse snRNA, female': (MOUSE_COLOR, (0, (1, 2)), 'o', False)}
label_spans = {sp: {s: np.quantile(POOLED_UNITS[(IS_HUMAN == (sp == 'human')) & (SEGMENT == s)], [.1, .9]) for s in SEGMENTS}
               for sp in ('mouse', 'human')}
grid_units = sc.to_segment_units(GRID, CUTS['pooled'])
BIN_ROWS = []
for g in LEAD_GENES:
    if g not in present:
        continue
    j = present.index(g)
    for n in NAMES:
        rows = np.flatnonzero(SPECIMEN == n)
        order_ = rows[np.argsort(POSITION[rows])]
        for chunk in np.array_split(order_, BINS_LEAD):
            counts_g = float(X_claim[chunk][:, j].sum())
            rate = counts_g / LIBRARY[chunk].sum() * 1e4
            BIN_ROWS.append({'gene': g, 'specimen': n, 'position': float(POSITION[chunk].mean()),
                             'segment units (pooled)': float(np.mean(POOLED_UNITS[chunk])), 'log CP10k': np.log(rate) if rate > 0 else np.nan})
LEAD_BINS = pd.DataFrame(BIN_ROWS)
LEAD_BINS.to_csv(tables / 'lead_gene_bins.csv', index=False)
fig = plt.figure(figsize=(180 * MM, 150 * MM))
gsp = fig.add_gridspec(4, 6, width_ratios=[3, 1.3] * 3, hspace=.75, wspace=.45)
for k, g in enumerate(LEAD_GENES):
    r_, c_ = divmod(k, 3)
    ax = fig.add_subplot(gsp[r_, 2 * c_])
    ax2 = fig.add_subplot(gsp[r_, 2 * c_ + 1])
    for sp, color, y0 in (('mouse', MOUSE_COLOR, .03), ('human', HUMAN_COLOR, .07)):
        for s in SEGMENTS:
            lo, hi = label_spans[sp][s]
            ax.plot([lo, hi], [y0, y0], transform=ax.get_xaxis_transform(), color=color, lw=1.5, alpha=.5, solid_capstyle='butt')
    for b in (1, 2):
        ax.axvline(b, color=LIGHT, lw=.6)
    if g in set(GENES):
        i = GENES.index(g)
        ax.plot(grid_units, MODEL['reference'][i] + np.log(1e4), color=MOUSE_COLOR, lw=1.2)
        ax.plot(grid_units, MODEL['case'][i] + np.log(1e4), color=HUMAN_COLOR, lw=1.2)
    for n in NAMES:
        b_ = LEAD_BINS[LEAD_BINS.gene.eq(g) & LEAD_BINS.specimen.eq(n)]
        ax.scatter(b_['segment units (pooled)'], b_['log CP10k'], s=3, lw=0, color=HUMAN_COLOR if n in HUMAN else MOUSE_COLOR,
                   marker='o' if n in (MOUSE[0], HUMAN[0]) else 's', alpha=.7)
    ax.set_xlim(0, 3)
    ax.set_xticks([.5, 1.5, 2.5], list(SEGMENTS))
    row = LEAD.loc[g]
    flags = ', '.join(f for f, on in (('label panel', row['label-panel gene']), ('anchor', row['axis anchor']),
                                      ('integration', row['integration feature'])) if on)
    klass = row['class (73)'] if isinstance(row['class (73)'], str) else 'unclassified'
    ax.set_title(f"{g} · {row['verdict']}\n{klass} · z top {100 * row['z percentile (top)']:.1f}%" + (f' · {flags}' if flags else ''),
                 loc='left', fontsize=5)
    if c_ == 0:
        ax.set_ylabel('log CP10k')
    for label, values in external_profiles(g).items():
        color, style, marker, filled = STYLE[label]
        ax2.plot(range(3), values, color=color, ls=style, lw=.8, marker=marker, ms=2.2, mfc=color if filled else 'white', mec=color, mew=.5)
    ax2.axhline(0, color=LIGHT, lw=.4)
    ax2.set_xticks(range(3), list(SEGMENTS), fontsize=4.5)
    ax2.tick_params(axis='y', labelsize=4.5)
fig.suptitle('Figure 5e · lead genes: NB group curves (71) with 20 equal-count specimen bins, on pooled segment units '
             '(bars: each species\' S1/S2/S3 label 10–90% spans); right: external S1/S2/S3 relative to S1 (log2). '
             'Between-segment reading only (C8, C14).', x=.01, ha='left', fontsize=5.5)
save(fig, 'fig5e_lead_genes')

# Supplementary: location agreement and the collection nulls.
fig, axes = plt.subplots(1, 2, figsize=(150 * MM, 55 * MM), layout='constrained')
jitter = np.random.default_rng(0).uniform(-.15, .15, len(CALLS))
axes[0].scatter(LOC['segment units (a, pooled)'], seg_index.to_numpy() + jitter, s=6, lw=0,
                color=np.where(LOC['direction (b)'].eq('human-high'), HUMAN_COLOR, MOUSE_COLOR))
for b in (1, 2):
    axes[0].axvline(b, color=LIGHT, lw=.6)
axes[0].set_yticks([0, 1, 2], list(SEGMENTS))
axes[0].set_xlabel('coordinate peak, pooled segment units (3.3 a)')
axes[0].set_ylabel('coordinate-free peak segment (3.3 b)')
axes[0].set_title(f"location: {LOCATION['segments agree (b vs a pooled)']} of {len(CALLS)} calls agree", loc='left')
run = RUNS[(('Census', 'male'), 'signed', 'matched')]
draws = pd.DataFrame(run['draws'], columns=run['collections'].astype(str))
observed = pd.Series(run['observed'], index=run['collections'].astype(str))
for name, color in (('calls', '#1b8a5a'), ('noise calls', MUTED)):
    axes[1].hist(draws[name], bins=60, color=color, alpha=.5, label=f'{name}: random collections')
    axes[1].axvline(observed[name], color=color, lw=1.2)
axes[1].legend()
axes[1].set_xlabel('mean joint replication z (Census cortex − male mouse)')
axes[1].set_title('overlap-preserving null (9,999 within-strata gene relabelings)', loc='left')
save(fig, 'supp_location_agreement_and_null')

fig, axes = plt.subplots(2, 5, figsize=(180 * MM, 70 * MM), layout='constrained', sharex=True)
for ax, g in zip(axes.ravel(), LEAD_GENES):
    if g in set(GENES):
        i = GENES.index(g)
        ax.plot(GRID, MODEL['reference'][i] + np.log(1e4), color=MOUSE_COLOR)
        ax.plot(GRID, MODEL['case'][i] + np.log(1e4), color=HUMAN_COLOR)
    for _, t in TRANSITIONS.iterrows():
        ax.axvline(t.position, color=HUMAN_COLOR if t.specimen in HUMAN else MOUSE_COLOR, lw=.4, ls='--' if t.transition == 'S1→S2' else ':')
    ax.set_title(g, loc='left')
fig.supxlabel("position (71's scale); dashed S1→S2, dotted S2→S3 per specimen")
save(fig, 'supp_lead_genes_coordinate')

## 11 · Comparison with the old results and the material-change rules (sections 6 and 7)

In [ ]:
old50 = pd.read_csv(OLD['50'] / 'tables' / 'table_replication_primary_overlap_null.csv')
old_ext = pd.read_csv(OLD['37'] / 'tables' / 'external_replication_summary.csv', index_col=0)
old_comp = pd.read_csv(OLD['37'] / 'tables' / 'table_S5_global_component.csv', index_col=0)
old_conv = pd.read_csv(OLD['50'] / 'tables' / 'conventional_only_overlap_null.csv', index_col=0)
old_list = pd.read_csv(OLD['50'] / 'tables' / 'primary_pathway_list.csv', index_col=0)
rep = REPLICATION.set_index(['reference', 'statistic', 'null'])
UNITS_H = LOCATION['median segment units (a pooled), human-high (a) — within-segment fraction not interpreted (C14)']
UNITS_M = LOCATION['median segment units (a pooled), mouse-high (a) — within-segment fraction not interpreted (C14)']
o50 = old50.set_index(['reference', 'statistic', 'null'])
CENSUS_M, LAKE_M = 'Census cortex − male mouse', 'Lake cortex − male mouse'
O_CENSUS, O_LAKE = 'Census cortex − male mouse (primary)', 'Lake/KPMP cortex − male mouse'


def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else np.nan


def clusters_of(ids):
    return set(REDUNDANCY.cluster_id.reindex([p for p in ids if p in SETS]).dropna())


COMPARE = [
    ('list', 'sets', '22 (12 core)', f'{len(CALLS)} ({len(CORE)} core)'),
    ('list', 'redundancy clusters of the list', f'{len(clusters_of(old22))} (71 clusters of the old 22)', str(len(clusters_of(CALLS)))),
    ('categories', 'mouse / shared / mixed / unresolved / human-led',
     '/'.join(str(int(old_list['notebook 43 category'].eq(c).sum())) for c in ('summary of mouse zonation', 'shared', 'mixed', 'unresolved'))
     + '/0', '/'.join(str(int(CAT.category.eq(c).sum())) for c in ('summary of mouse zonation', 'shared', 'mixed', 'unresolved', 'human-led'))),
    ('direction', 'human-high / mouse-high', '10 / 12', f"{LOCATION['human-high (b)']} / {LOCATION['mouse-high (b)']} (b)"),
    ('location', 'median peak (segment units), human-high / mouse-high', '0.50 / 1.95 (SCF13; DPT13 0.95 / 1.77)',
     f"segment index {LOCATION['median segment index, human-high (b)']} / {LOCATION['median segment index, mouse-high (b)']} (b); "
     f"units {UNITS_H:.2f} / {UNITS_M:.2f} (a pooled, not interpreted within segment)"),
    ('location', 'segment agreement between the two rulers', '15 of 22 (SCF13 vs DPT13)',
     f"{LOCATION['segments agree (b vs a pooled)']} of {len(CALLS)} (b vs a pooled)"),
    ('support', 'called by the coordinate-free step screen', '20 of 22 (Gaussian 45)', f"{int(ROB['segment F-test (75)'].sum())} of {len(CALLS)} (NB segment F-test, 75)"),
    ('support', 'spline-based steps', '8 of 22', f"{int(ROB['steps on the spline (74)'].sum()) if HAS74 else 'n/a'} of {len(CALLS)} (74)"),
    ('support', 'beyond S1/S2/S3 steps', '22 of 22', f"{int(ROB['beyond label steps, test A (74)'].sum()) if HAS74 else 'n/a'} of {len(CALLS)} (74 test A)"),
    ('robustness', 'detected in both (effect, q ≤ 0.10)', '20 / 14 of 22', f"{int(ROB['retained: detected in both (≥ 5% each species)'].sum())} of {len(CALLS)}"),
    ('robustness', 'sex-unbiased', '21 / 20 of 22', f"{int(ROB['retained: mouse sex-unbiased'].sum())} of {len(CALLS)}"),
    ('robustness', 'equal probes: lose significance', '7 of 22', f"{int(ROB['probe-sensitive'].sum())} of {len(CALLS)} not retained"),
    ('replication', 'mean z, Census / Lake (male)', f"{o50.loc[(O_CENSUS, 'signed', 'matched'), 'mean z, primary (22)']:.2f} / {o50.loc[(O_LAKE, 'signed', 'matched'), 'mean z, primary (22)']:.2f}",
     f"{rep.loc[(CENSUS_M, 'signed', 'matched'), 'mean z, calls']:.2f} / {rep.loc[(LAKE_M, 'signed', 'matched'), 'mean z, calls']:.2f}"),
    ('replication', 'p_random, Census / Lake', '0.0001 / 0.0001',
     f"{rep.loc[(CENSUS_M, 'signed', 'matched'), 'p_random, calls']:.4f} / {rep.loc[(LAKE_M, 'signed', 'matched'), 'p_random, calls']:.4f}"),
    ('replication', 'p vs comparator, Census / Lake', '0.004 / 0.007 (relabel-called, 21)',
     f"{rep.loc[(CENSUS_M, 'signed', 'matched'), 'p calls vs noise calls']:.4f} / {rep.loc[(LAKE_M, 'signed', 'matched'), 'p calls vs noise calls']:.4f} (noise calls)"),
    ('replication', 'programs, label permutation, Census / Lake', '0.039 / 0.065',
     f"{rep.loc[(CENSUS_M, 'signed', 'matched'), 'p programs (label permutation)']:.4f} / {rep.loc[(LAKE_M, 'signed', 'matched'), 'p programs (label permutation)']:.4f}"),
    ('replication', 'decision, Census / Lake', 'specific / better than random only',
     f"{DECISIONS.loc[CENSUS_M, 'decision (calls)']} / {DECISIONS.loc[LAKE_M, 'decision (calls)']}"),
    ('replication', 'slope-free p vs comparator, Census / Lake', '0.26 / 0.13',
     f"{rep.loc[(CENSUS_M, 'slope-free', 'matched'), 'p calls vs noise calls']:.4f} / {rep.loc[(LAKE_M, 'slope-free', 'matched'), 'p calls vs noise calls']:.4f}"),
    ('replication', 'per call, BH ≤ 0.10 within list: Census / Lake (male); (female)', '12 / 5 of 22; 5 / 3',
     f"{PER_CALL_COUNTS.loc['replicated (BH ≤ 0.10 within list)', CENSUS_M]} / {PER_CALL_COUNTS.loc['replicated (BH ≤ 0.10 within list)', LAKE_M]} of {len(CALLS)}; "
     f"{PER_CALL_COUNTS.loc['replicated (BH ≤ 0.10 within list)', 'Census cortex − female mouse']} / {PER_CALL_COUNTS.loc['replicated (BH ≤ 0.10 within list)', 'Lake cortex − female mouse']}"),
    ('replication', 'gene ρ top 10%: Census / Lake (random median; top-expressed)',
     f"{old_ext.loc['gene ρ, top 10% T_spatial', O_CENSUS]:.2f} / {old_ext.loc['gene ρ, top 10% T_spatial', O_LAKE]:.2f} "
     f"({old_ext.loc['gene ρ, expression-matched random (median)', O_CENSUS]:.2f} / {old_ext.loc['gene ρ, expression-matched random (median)', O_LAKE]:.2f}; "
     f"{old_ext.loc['gene ρ, top 10% mean expression', O_CENSUS]:.2f} / {old_ext.loc['gene ρ, top 10% mean expression', O_LAKE]:.2f})",
     f"{GENE_LEVEL.loc['gene ρ, top 10% of the score', CENSUS_M]:.2f} / {GENE_LEVEL.loc['gene ρ, top 10% of the score', LAKE_M]:.2f} "
     f"({GENE_LEVEL.loc['gene ρ, expression-matched random (median)', CENSUS_M]:.2f} / {GENE_LEVEL.loc['gene ρ, expression-matched random (median)', LAKE_M]:.2f}; "
     f"{GENE_LEVEL.loc['gene ρ, top 10% mean expression', CENSUS_M]:.2f} / {GENE_LEVEL.loc['gene ρ, top 10% mean expression', LAKE_M]:.2f})"),
    ('replication', 'global slope: ours / Census / Lake',
     f"{old_comp.loc['slope of interaction on mouse gradient'].iloc[0]:.2f} / {old_comp.loc['slope of interaction on mouse gradient', O_CENSUS]:.2f} / "
     f"{old_comp.loc['slope of interaction on mouse gradient', O_LAKE]:.2f}",
     f"{COMPONENT.loc['slope of interaction on mouse gradient'].iloc[0]:.2f} / {COMPONENT.loc['slope of interaction on mouse gradient', CENSUS_M]:.2f} / "
     f"{COMPONENT.loc['slope of interaction on mouse gradient', LAKE_M]:.2f}"),
    ('conventional-only', 'sets; MW p; D p (Census / Lake); decision',
     f"{int(old_conv.loc['conventional-only pathways tested', O_CENSUS])}; {old_conv.loc['Mann–Whitney p (overlap-naive, as notebook 37)', O_CENSUS]:.1e}; "
     f"{old_conv.loc['p, D under joint relabeling', O_CENSUS]:.2f} / {old_conv.loc['p, D under joint relabeling', O_LAKE]:.2f}; not supported",
     f"{int(CONVENTIONAL_TABLE.loc['conventional-only sets tested', 'Census cortex − male mouse'])}; "
     f"{CONVENTIONAL_TABLE.loc['Mann–Whitney p (overlap-naive, as notebook 37)', 'Census cortex − male mouse']:.1e}; "
     f"{CONVENTIONAL_TABLE.loc['p, D under joint relabeling', 'Census cortex − male mouse']:.2f} / "
     f"{CONVENTIONAL_TABLE.loc['p, D under joint relabeling', 'Lake cortex − male mouse']:.2f}; {CONVENTIONAL_DECISION}")]
for g in ('Gatm', 'Acadm', 'Acaa2', 'Dcxr', 'Ugt3a1', 'Rbp4'):
    r = LEAD.loc[g]
    COMPARE.append(('lead genes', f'{g}: rank top %, ours human S3 − S1, class, verdict',
                    f"{100 * r['old T_spatial percentile (37)']:.2f}% (T_spatial); {r.get('old ours human S3 − S1', np.nan):+.2f}; "
                    f"{r['old class (43)'] if isinstance(r['old class (43)'], str) else 'unclassified'}; {r['old verdict']}",
                    f"{100 * r['z percentile (top)']:.2f}% (z); {r.get('ours human S3 − S1 (mt-free)', np.nan):+.2f}; "
                    f"{r['class (73)'] if isinstance(r['class (73)'], str) else 'unclassified'}; {r['verdict']}"))
for story in LEAD_STORIES:
    COMPARE.append(('lead stories', story, STORIES.loc[story, 'old verdict'], STORIES.loc[story, 'verdict']))
for label, ids in (('old 22', old22), ('old 36', old36), ('old 60', old60), ("69's 97", old97)):
    COMPARE.append(('overlap', f'calls ∩ {label} (sets; Jaccard over sets / clusters)',
                    f'{len(set(ids))}', f"{len(set(CALLS) & set(ids))}; {jaccard(CALLS, ids):.2f} / {jaccard(clusters_of(CALLS), clusters_of(ids)):.2f}"))
COMPARE = pd.DataFrame(COMPARE, columns=['part', 'quantity', 'old', 'new'])
COMPARE.to_csv(tables / 'compare_old.csv', index=False)
display(COMPARE)
print('Old 22 not called:', '; '.join(PATHWAY_NAME.get(p, p) for p in old22 if p not in set(CALLS)))
print('New calls outside the old 36:', len([p for p in CALLS if p not in set(old36)]))

resolved_n = int(CAT.category.ne('unresolved').sum())
mouse_share = CAT.category.eq('summary of mouse zonation').sum() / resolved_n if resolved_n else np.nan
med_h, med_m = LOCATION['median segment index, human-high (b)'], LOCATION['median segment index, mouse-high (b)']
agree_share = LOCATION['segments agree (b vs a pooled)'] / len(CALLS)
prand = [rep.loc[(ref, 'signed', 'matched'), 'p_random, calls'] for ref in (CENSUS_M, LAKE_M)]
slope_p = [rep.loc[(ref, 'slope-free', 'matched'), 'p calls vs noise calls'] for ref in (CENSUS_M, LAKE_M)]
story_changed = [s for s in LEAD_STORIES if STORIES.loc[s, 'verdict'] != STORIES.loc[s, 'old verdict']]
rank_fail = ([g for g in ['Gatm'] if not LEAD.loc[g, 'z percentile (top)'] <= .005]
             + [g for g in ['Acadm', 'Acaa2', 'Dcxr', 'Ugt3a1'] if not LEAD.loc[g, 'z percentile (top)'] <= .02])
reversal_lost = [g for g in ['Acaa2', 'Dcxr', 'Ugt3a1'] if LEAD.loc[g, 'class (73)'] != 'reversal']
MATERIAL = pd.DataFrame([
    ('M1 categories', '13 of 17 resolved; 0 human-led',
     f"{int(CAT.category.eq('summary of mouse zonation').sum())} of {resolved_n} resolved ({mouse_share:.2f}); "
     f"{int(CAT.category.eq('human-led').sum())} human-led", bool(mouse_share < .5 or CAT.category.eq('human-led').any())),
    ('M2 direction by segment (b)', 'human-high S1; mouse-high S2–S3',
     f'median segment index human-high {med_h}, mouse-high {med_m}', bool((med_h != 0) or (med_m == 0))),
    ('M3 location reliability', '15 of 22 (0.68)', f"{LOCATION['segments agree (b vs a pooled)']} of {len(CALLS)} ({agree_share:.2f})",
     bool(agree_share < .5)),
    ('M4 replication', 'Census specific; p_random 0.0001 both',
     f"Census {DECISIONS.loc[CENSUS_M, 'decision (calls)']}; p_random {prand[0]:.4f} / {prand[1]:.4f}",
     bool(DECISIONS.loc[CENSUS_M, 'decision (calls)'] != 'specific' or max(prand) > .01)),
    ('M5 beyond flattening', 'slope-free p 0.26 / 0.13', f'{slope_p[0]:.4f} / {slope_p[1]:.4f}', bool(min(slope_p) <= .05)),
    ('M6 lead genes', 'creatine lead, fao lead, detox supporting, reversal lead; Gatm top 0.14%, others ≤ 1.05%; Acaa2/Dcxr/Ugt3a1 reversal',
     f"stories changed: {story_changed or 'none'}; ranks outside rule: {rank_fail or 'none'}; reversal lost: {reversal_lost or 'none'}",
     bool(story_changed or rank_fail or reversal_lost)),
    ('M7 conventional-only', 'group claim not supported once overlap is respected', CONVENTIONAL_DECISION,
     CONVENTIONAL_DECISION != 'group claim not supported once overlap is respected')],
    columns=['rule', 'old', 'new', 'material'])
MATERIAL.to_csv(tables / 'material_change.csv', index=False)
display(MATERIAL)

## 12 · How the protocol was read

In [ ]:
READING.append("M2 reads 'the median peak segment' as the median of the segment index (0, 1, 2) over the calls of each "
               'direction; a median between two segments counts as "not S1" for human-high calls.')
READING.append("M6 reads 'outside the top 0.5% / 2%' as a percentile rank of 71's condition z above 0.005 / 0.02 among the "
               "universe; the old percentiles were Gaussian T_spatial ranks (notebook 37).")
READING.append('Collection nulls run with seed 50 (notebook 50); the gene-level random draws with seed 12 (notebook 37); '
               "the attenuation bootstrap with seed 38 (notebook 38).")
for k, line in enumerate(READING, 1):
    print(f'{k}. {line}')
pd.Series(READING, name='reading').to_csv(tables / 'how_the_protocol_was_read.csv', index=False)

## 13 · Summary and run record

In [ ]:
SUMMARY = {
    'calls': len(CALLS), 'redundancy clusters': len(clusters_of(CALLS)), 'programs': int(PROGRAM.nunique()), 'core calls': len(CORE),
    'categories': CAT.category.value_counts().to_dict(), 'direction (b)': LOC['direction (b)'].value_counts().to_dict(),
    'segment × direction (b)': SEG_BY_DIR.to_dict(), 'location agreement (b vs a pooled)': LOCATION['segments agree (b vs a pooled)'],
    'segment F-test support': int(ROB['segment F-test (75)'].sum()),
    'replication decisions': DECISIONS['decision (calls)'].to_dict(), 'conventional-only': CONVENTIONAL_DECISION,
    'lead stories': STORIES['verdict'].reindex(LEAD_STORIES).to_dict(),
    'material changes': MATERIAL.set_index('rule').material.to_dict(), 'attenuation': ATTENUATION,
    'noise-call comparator': len(NOISE), 'heatmap program': PROGRAM_SUMMARY.loc[HEAT_PROGRAM, 'name']}
print(json.dumps(SUMMARY, indent=2, default=str))
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'protocol': 'docs/results/pt-package-species-calls.md',
          'package_provenance': {k: PROV[k] for k in ('notebook', 'notebook_logic_version', 'package', 'run_folder')},
          'pass1_fingerprint': FINGERPRINT, 'guards': GUARDS, 'reading': READING, 'summary': SUMMARY,
          'seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f'Done in {record["seconds"]:.0f} s')